# GitHub 최종 코드·데이터로 Streamlit 실행
- GPU/설정 → GitHub 받기 → 설치·검수 데이터 준비 → 앱 실행 순서
- 실제 모델 점검 셀은 선택 실행이며 모델·추출 코드 변경 후 재점검 권장
- 기본 브랜치는 main이며 병합 전 화면 점검 시 BRANCH를 feat/streamlit-ui로 변경
- Gemini 응답·시간 확인은 설치 후 마지막 코드 셀에서 선택 실행
- 원문 데이터 변경 없이 Q1 순위 검수·Q2 동일 사업 출처 자료를 생성·검증함
- 기본 접속은 Colab 프록시이며 공개 터널 셀은 필요한 경우만 실행


In [ ]:
import subprocess, sys, os, time
from pathlib import Path
import torch

assert torch.cuda.is_available(), "런타임 → 런타임 유형 변경 → T4 GPU 를 고른 뒤 다시 실행해 주세요."
print("GPU:", torch.cuda.get_device_name(0))

os.environ["BANGEOL_EMBED_BATCH_SIZE"] = "4"        # T4 메모리용 (팀원 설정)
os.environ["BANGEOL_REQUIRE_REVIEW_DATA"] = "1"
os.environ["BANGEOL_PIPELINE_MODE"] = "실제 모델"     # 앱을 열면 바로 실제 모델

REPO = "lcsvvo/Isolated-Youth-Welfare-Access-AI"
BRANCH = "main"
ROOT = Path("/content/Isolated-Youth-Welfare-Access-AI")

from google.colab import userdata
try:                                                  # 일상 대화용 키는 선택 사항 (없어도 오류 없이 넘어감)
    _key = userdata.get("GEMINI_API_KEY")
except Exception:
    _key = None
if _key:
    os.environ["GEMINI_API_KEY"] = _key
    print("Gemini 키: 있음")
else:
    os.environ.pop("GEMINI_API_KEY", None)
    print("Gemini 키: 없음 (일상 대화는 기본 답으로 나와요)")
del _key

# 1, 2, 3, 5 셀 돌리면 됨

In [ ]:
import base64
from google.colab import userdata
try:
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
if not tok:
    raise RuntimeError("보안 비밀의 GH_TOKEN과 노트북 액세스를 확인해 주세요.")
# 토큰을 원격 URL이나 git 인수에 넣지 않음
_git_env = os.environ.copy()
_git_env.update({"GIT_CONFIG_COUNT": "2",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": "AUTHORIZATION: basic " + base64.b64encode(("x-access-token:" + tok).encode()).decode(),
    "GIT_CONFIG_KEY_1": "credential.helper", "GIT_CONFIG_VALUE_1": ""})
def git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, env=_git_env, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError("GitHub 코드를 받지 못했어요. 접근 권한·브랜치·로컬 변경사항을 확인해 주세요.")
    return r.stdout.strip()
try:
    if not ROOT.exists():
        git("clone", "--single-branch", "-b", BRANCH, f"https://github.com/{REPO}.git", str(ROOT))
    else:
        remote = git("remote", "get-url", "origin", cwd=ROOT)
        if remote != f"https://github.com/{REPO}.git":
            raise RuntimeError("기존 폴더의 원격 저장소를 확인하거나 새 ROOT 폴더를 사용해 주세요.")
        changes = git("status", "--porcelain", "--untracked-files=no", cwd=ROOT)
        if changes:
            raise RuntimeError("런타임에 수정된 파일이 있어 업데이트를 멈췄어요. 수정사항을 보존하거나 새 ROOT 폴더를 사용해 주세요.")
        git("fetch", "origin", f"{BRANCH}:refs/remotes/origin/{BRANCH}", cwd=ROOT)
        exists = subprocess.run(["git", "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"], cwd=ROOT).returncode == 0
        if exists:
            git("switch", BRANCH, cwd=ROOT)
        else:
            git("switch", "--track", "-c", BRANCH, f"origin/{BRANCH}", cwd=ROOT)
        git("merge", "--ff-only", f"origin/{BRANCH}", cwd=ROOT)
    print("실행 코드:", git("log", "-1", "--format=%h %s", cwd=ROOT))
finally:
    del tok, _git_env


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "ai/requirements-streamlit.txt")], check=True)
# GitHub에 있는 원본·최종 검수값으로 추천/출처 데이터를 생성하고 버전을 검증
subprocess.run([sys.executable, str(ROOT / "ai/scripts/prepare_runtime.py")], cwd=ROOT, check=True)
print("설치 및 검수 데이터 준비 완료")


In [ ]:
check = subprocess.Popen([sys.executable, "-u", "scripts/real_model_check.py", "--explain"],
                         cwd=ROOT / "ai", env=os.environ.copy(),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in check.stdout:
    print(line, end="")
assert check.wait() == 0, "실제 모델 점검 실패: 위 오류를 붙여 주세요."
print("\n실제 모델 점검 통과")
# 건너뛰어도 됨 처음 한 번만 하면

In [ ]:
import runpy
launch = runpy.run_path(str(ROOT / "ai/scripts/colab_launch.py"))
st_proc = launch["start_preview"](ROOT, globals().get("st_proc"))


In [ ]:
# 주의: 이 주소는 링크만 알면 누구나 접속 가능 → 실제 개인정보 입력 금지
# 아래에 나온 링크로 들어가기!
import re
CF = Path("/content/cloudflared")

# 1) 파일이 없을 때만 내려받기 (오류 내용 보이게)
if not CF.exists():
    r = subprocess.run(["wget", "-nv", "-O", str(CF),
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        CF.unlink(missing_ok=True)
        raise RuntimeError("cloudflared 내려받기 실패: 위 메시지를 붙여 주세요.")
    CF.chmod(0o755)

# 2) 예전 터널 끄고 새로 켜기
subprocess.run(["pkill", "-f", "cloudflared tunnel"])
time.sleep(2)
cf_log = open("/content/cf.log", "w")
cf = subprocess.Popen([str(CF), "tunnel", "--url", "http://localhost:8501"],
                      stdout=cf_log, stderr=subprocess.STDOUT)

# 3) 주소 나올 때까지 최대 30초 기다리기
for _ in range(30):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read())
    if m:
        print("접속 주소:", m.group(0))
        break
    time.sleep(1)
else:
    print(open("/content/cf.log").read()[-2000:])
    raise RuntimeError("주소를 못 받았어요: 위 로그를 붙여 주세요.")

In [ ]:
# Gemini 연결 확인: 합성 인사 한 문장만 전송
probe = '''
from time import perf_counter
import chitchat

started = perf_counter()
result = chitchat.reply("안녕하세요")
elapsed = perf_counter() - started
if result.source == "gemini":
    print(f"Gemini 응답 성공 ({elapsed:.2f}초)")
else:
    print(f"Gemini 기본 답변 사용 ({result.error}, {elapsed:.2f}초)")
'''
subprocess.run([sys.executable, "-c", probe], cwd=ROOT / "ai", check=True)
subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"])